# Notebook 02 — SFT Trials

**Inputs:** `data/test_prompts.json`, `data/gold_answers.json`, `configs/sft_trials.json`  
**Outputs:** `results/sft_<name>.json` × 5, `results/sft_winner.json`, `models/sft_<name>/` × 5  
**Runtime:** ~3–8 hours on Kaggle T4

---
⚠ **Crash recovery:** Already-finished trials are skipped automatically on re-run.

⚠ **Parallel mode:** To split across two Kaggle sessions, set `TRIALS_TO_RUN` to
`[0, 1, 2]` in Session A and `[3, 4]` in Session B. Set `SELECT_WINNER = False`
in both, then re-run either session (with all indices) once both finish.

## Cell 1 — Configuration (edit here for parallel sessions)

In [ ]:
# ─── EDIT THESE TWO LINES FOR PARALLEL SESSIONS ───────────────────────────
TRIALS_TO_RUN = [1]   # Change to [0,1,2] or [3,4] for parallel split
SELECT_WINNER = False               # Set False when running a subset
# ───────────────────────────────────────────────────────────────────────────

print(f"Will run trial indices: {TRIALS_TO_RUN}")
print(f"Select winner at end:   {SELECT_WINNER}")

## Cell 2 — Install dependencies

In [ ]:
import subprocess, sys

PACKAGES = [
    "transformers>=4.45.0", "peft>=0.13.0", "trl>=0.11.0",
    "bitsandbytes>=0.44.0", "accelerate>=1.0.0", "datasets>=3.0.0",
    "sacrebleu", "bert-score",
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers", "huggingface_hub"], check=False)
print("Done.")

## Cell 3 — Paths & HF login

In [ ]:
import os, sys, json, gc, time
from pathlib import Path
import torch

KAGGLE = Path("/kaggle").exists()
if KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working/daa-helper")
    if not PROJECT_ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone",
                        "https://github.com/mimadraza/daa-helper.git",
                        str(PROJECT_ROOT)], check=False)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste HF token: ").strip()
login(token=HF_TOKEN)

HF_USERNAME = "mimadraza"
BASE_MODEL  = "TinyLlama/TinyLlama_v1.1"

from utils.io_helpers import load_json, save_trial_result, list_existing_results
from utils.evaluation  import run_inference_on_prompts, evaluate_responses, free_memory
print("Setup complete.")

## Cell 4 — Load configs, prompts, gold answers

In [ ]:
sft_config   = load_json("sft_trials.json",  base_dir="configs")
prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data    = load_json("gold_answers.json", base_dir="data")

prompts      = [p["prompt"]      for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]
all_trials   = sft_config["trials"]

# Apply subset filter from Cell 1
trials = [all_trials[i] for i in TRIALS_TO_RUN if i < len(all_trials)]

print(f"Running {len(trials)} of {len(all_trials)} trials:")
for t in trials:
    print(f"  {t['name']}: rank={t['lora_r']}, target={t['target_modules']}, lr={t['learning_rate']}")

## Cell 5 — Load SFT dataset

In [ ]:
from datasets import load_dataset

print("Loading open-r1/codeforces-cots (solutions_w_editorials)...")
raw_ds = load_dataset("open-r1/codeforces-cots", "solutions_w_editorials", split="train")
print(f"Full dataset: {len(raw_ds)} samples | Columns: {raw_ds.column_names}")

N_TRAIN, N_EVAL = 3000, 200
raw_ds   = raw_ds.shuffle(seed=42)
train_ds = raw_ds.select(range(min(N_TRAIN, len(raw_ds))))
eval_ds  = raw_ds.select(range(N_TRAIN, min(N_TRAIN + N_EVAL, len(raw_ds))))
train_ds = train_ds.rename_column("generation", "completion")
eval_ds  = eval_ds .rename_column("generation", "completion")

assert "messages" in train_ds.column_names, \
    f"Expected 'messages' column, got: {train_ds.column_names}"

print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

## Cell 6 — Define training & evaluation helpers

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                           BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl  import SFTTrainer, SFTConfig


def load_base_model_for_training():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto",
        use_safetensors=False, trust_remote_code=True, dtype=torch.bfloat16,
    )
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer


def run_sft_trial(trial_cfg, train_dataset, eval_dataset, output_dir):
    print(f"\n{'='*70}\nSFT TRIAL: {trial_cfg['name']}\n{'='*70}")
    model, tokenizer = load_base_model_for_training()

    peft_config = LoraConfig(
        r=trial_cfg["lora_r"], lora_alpha=trial_cfg["lora_alpha"],
        lora_dropout=trial_cfg.get("lora_dropout", 0.05),
        target_modules=trial_cfg["target_modules"],
        bias="none", task_type="CAUSAL_LM",
    )
    sft_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=trial_cfg["num_train_epochs"],
        per_device_train_batch_size=trial_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=trial_cfg["gradient_accumulation_steps"],
        per_device_eval_batch_size=trial_cfg["per_device_train_batch_size"],
        learning_rate=trial_cfg["learning_rate"],
        logging_steps=10, eval_strategy="steps",
        max_length=trial_cfg.get("max_seq_length", 1024),
        eval_steps=100, save_strategy="no", warmup_steps=5,
        lr_scheduler_type="cosine", bf16=True, optim="paged_adamw_8bit",
        report_to="none", gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataset_text_field=None, packing=False,
    )
    trainer = SFTTrainer(
        model=model, args=sft_args,
        train_dataset=train_dataset, eval_dataset=eval_dataset,
        peft_config=peft_config, processing_class=tokenizer,
    )
    start        = time.time()
    train_result = trainer.train()
    elapsed      = time.time() - start

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    eval_metrics = trainer.evaluate()

    train_metrics = {
        "train_loss": train_result.training_loss,
        "eval_loss":  eval_metrics.get("eval_loss"),
        "train_runtime_seconds": elapsed,
        "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
        "global_step": train_result.global_step,
    }
    del trainer, model
    free_memory()
    return output_dir, train_metrics


def evaluate_adapter(adapter_dir, prompts, gold_answers):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base  = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb, device_map="auto",
        trust_remote_code=True, dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    responses    = run_inference_on_prompts(model, tokenizer, prompts,
                                            max_new_tokens=512, temperature=0.7)
    eval_results = evaluate_responses(responses, gold_answers)
    samples      = [{"prompt": p, "response": r, "gold": g}
                    for p, r, g in zip(prompts, responses, gold_answers)]
    del model, base
    free_memory()
    return eval_results, samples


print("Helpers defined.")

## Cell 7 — Run trials

Already-finished trials are skipped automatically.

In [ ]:
existing = set(list_existing_results(stage="sft"))
print(f"Existing SFT results: {existing or '(none)'}")

for trial in trials:
    result_filename = f"sft_{trial['name']}.json"
    if result_filename in existing:
        print(f"\n✓ Skipping {trial['name']} (already completed)")
        continue

    output_dir = str(PROJECT_ROOT / "models" / f"sft_{trial['name']}")
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    try:
        adapter_dir, train_metrics = run_sft_trial(trial, train_ds, eval_ds, output_dir)

        print("\nEvaluating on 10 test prompts...")
        eval_results, sample_responses = evaluate_adapter(adapter_dir, prompts, gold_answers)

        save_trial_result(
            trial_name=trial["name"], stage="sft", config=trial,
            eval_results=eval_results, train_metrics=train_metrics,
            sample_responses=sample_responses,
        )

        agg = eval_results["aggregate"]
        print(f"\n✓ {trial['name']}")
        print(f"  Mean BLEU:         {agg['mean_bleu']:.2f}")
        print(f"  Mean BERTScore F1: {agg['mean_bertscore_f1']:.4f}")
        print(f"  Combined score:    {agg['combined_score']:.4f}")
        print(f"  Train loss:        {train_metrics['train_loss']:.4f}")
        print(f"  Eval loss:         {train_metrics['eval_loss']:.4f}")
        print(f"  Time:              {train_metrics['train_runtime_seconds']:.0f}s")

    except Exception as e:
        import traceback
        print(f"\n✗ ERROR in {trial['name']}: {e}")
        traceback.print_exc()
        free_memory()

print("\n\nAll assigned trials done.")

## Cell 8 — Select winner

Skip this cell if `SELECT_WINNER = False` (parallel subset mode).

In [ ]:
if not SELECT_WINNER:
    print("SELECT_WINNER=False — skipping. Re-run with SELECT_WINNER=True once all sessions finish.")
else:
    results = []
    for trial in all_trials:
        try:
            results.append(load_json(f"sft_{trial['name']}.json", base_dir="results"))
        except FileNotFoundError:
            print(f"  ⚠ Missing result for {trial['name']}")

    results.sort(key=lambda r: (
        -r["evaluation"]["aggregate"]["combined_score"],
         r["training_metrics"].get("eval_loss") or float("inf"),
    ))

    print("=" * 70)
    print("SFT TRIAL RANKING (best first)")
    print("=" * 70)
    print(f"{'Trial':<30} {'BLEU':>8} {'BERTScore':>10} {'Combined':>10} {'EvalLoss':>10}")
    for r in results:
        a  = r["evaluation"]["aggregate"]
        el = r["training_metrics"].get("eval_loss", float("nan"))
        print(f"{r['trial_name']:<30} {a['mean_bleu']:>8.2f} "
              f"{a['mean_bertscore_f1']:>10.4f} {a['combined_score']:>10.4f} {el:>10.4f}")

    winner = results[0]
    print(f"\n🏆 WINNER: {winner['trial_name']}")

    with open(PROJECT_ROOT / "results" / "sft_winner.json", "w") as f:
        json.dump({
            "winning_trial":   winner["trial_name"],
            "winning_config":  winner["config"],
            "winning_metrics": winner["evaluation"]["aggregate"],
        }, f, indent=2)
    print("Saved → results/sft_winner.json")

## Cell 9 — Push results to GitHub & adapter to HF Hub

In [ ]:
import subprocess

if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    GITHUB_TOKEN = sec.get_secret("GITHUB_TOKEN")
    GITHUB_USER  = sec.get_secret("GITHUB_USER")

    cmds = [
        ["git", "config", "--global", "user.email", "you@example.com"],
        ["git", "config", "--global", "user.name", GITHUB_USER],
        ["git", "-C", str(PROJECT_ROOT), "add", "results/"],
        ["git", "-C", str(PROJECT_ROOT), "commit", "-m", "sft results"],
        ["git", "-C", str(PROJECT_ROOT), "push",
         f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/daa-helper.git"],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True)
        out = (r.stdout or r.stderr).strip()
        if out: print(out)

    # Push winning adapter to HF Hub
    if SELECT_WINNER and 'winner' in dir():
        from utils.io_helpers import push_to_hf_hub
        winner_dir = str(PROJECT_ROOT / "models" / f"sft_{winner['trial_name']}")
        push_to_hf_hub(
            local_dir=winner_dir,
            repo_id=f"{HF_USERNAME}/daa-helper-tinyllama-sft-winner",
            repo_type="model",
            commit_message=f"SFT winner: {winner['trial_name']}",
        )
        print(f"Pushed adapter → {HF_USERNAME}/daa-helper-tinyllama-sft-winner")
else:
    print("Not on Kaggle — skipping push.")

print("\n✓ Notebook 02 complete. Run Notebook 03 next.")